# 스트리밍 중간에 툴 요청이 온다? — 직접 확인하기

"텍스트 응답이 끝나기 전에 툴 사용 요청이 오면 중간 스트리밍으로 처리한다"는 얘기가 맞는지 확인하는 노트북입니다.

**결론부터**: 맞습니다. 어시스턴트 응답 하나는 여러 콘텐츠 블록의 리스트이고, 스트리밍에서는
블록이 **생성되는 순서대로** 내려옵니다. 모델이 "날씨를 확인해볼게요"라는 텍스트를 먼저 만들고
이어서 툴 호출을 만들면, 클라이언트는 **하나의 스트림 안에서** 이 순서 그대로 받습니다:

```
content_block_start (index=0, type=text)
  text_delta "날씨를" → "확인해" → "볼게요"     ← 텍스트가 먼저 흘러나옴
content_block_stop  (index=0)
content_block_start (index=1, type=tool_use)    ← 스트림이 안 끝났는데 툴 요청 시작
  input_json_delta '{"city":' → '"서울"}'        ← 툴 입력 JSON도 조각으로 스트리밍
content_block_stop  (index=1)
message_delta (stop_reason="tool_use")
```

Claude Code가 텍스트를 실시간으로 보여주다가 곧바로 툴 실행 표시로 넘어가는 게 바로 이 구조입니다.
이 노트북에서 (1) 이벤트 타임라인을 밀리초 단위로 찍어 순서를 증명하고,
(2) 스트리밍 + 툴 실행을 결합한 완전한 에이전틱 루프를 만들어봅니다.

## 1. 클라이언트 초기화 + 툴 준비

In [1]:
import json
import os
import time

import anthropic
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("ANTHROPIC_API_KEY"), ".env 파일에 ANTHROPIC_API_KEY를 설정해주세요"

client = anthropic.Anthropic()
MODEL = "claude-opus-4-8"


def get_weather(city: str) -> str:
    data = {"서울": {"temp": 31, "sky": "맑음"}, "부산": {"temp": 28, "sky": "흐림"}}
    info = data.get(city, {"temp": 25, "sky": "정보 없음"})
    return json.dumps({"city": city, **info}, ensure_ascii=False)


def get_air_quality(city: str) -> str:
    data = {"서울": {"pm10": 45, "grade": "보통"}, "부산": {"pm10": 22, "grade": "좋음"}}
    info = data.get(city, {"pm10": None, "grade": "정보 없음"})
    return json.dumps({"city": city, **info}, ensure_ascii=False)


TOOL_FUNCTIONS = {"get_weather": get_weather, "get_air_quality": get_air_quality}

tools = [
    {
        "name": "get_weather",
        "description": "도시의 현재 날씨(기온, 하늘 상태)를 조회합니다.",
        "input_schema": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "도시 이름 (예: 서울)"}},
            "required": ["city"],
        },
    },
    {
        "name": "get_air_quality",
        "description": "도시의 현재 미세먼지(PM10) 수치와 등급을 조회합니다.",
        "input_schema": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "도시 이름 (예: 서울)"}},
            "required": ["city"],
        },
    },
]

# 툴 호출 전에 텍스트를 먼저 말하도록 유도 — Claude Code도 이런 식으로 진행 안내를 시킵니다
SYSTEM = "도구를 호출하기 전에, 지금 무엇을 확인하려는지 사용자에게 한 문장으로 먼저 말한 뒤 도구를 호출하세요."

print("준비 완료")

준비 완료


## 2. 이벤트 타임라인 — 텍스트가 먼저, 툴 요청이 뒤에, 같은 스트림에서

스트림의 모든 이벤트를 경과 시간(ms)과 함께 출력합니다.
`text_delta`가 먼저 흘러나오다가, **스트림이 끝나기 전에** `tool_use` 블록이 시작되는 것을 직접 확인하세요.

In [2]:
start = time.perf_counter()


def ts() -> str:
    return f"{(time.perf_counter() - start) * 1000:7.0f}ms"


with client.messages.stream(
    model=MODEL,
    max_tokens=8192,
    system=SYSTEM,
    tools=tools,
    messages=[{"role": "user", "content": "서울 날씨랑 미세먼지 알려줘."}],
) as stream:
    for event in stream:
        if event.type == "message_start":
            print(ts(), "▶ message_start")
        elif event.type == "content_block_start":
            cb = event.content_block
            extra = f" name={cb.name}" if cb.type == "tool_use" else ""
            print(ts(), f"▶ content_block_start  index={event.index} type={cb.type}{extra}")
        elif event.type == "content_block_delta":
            if event.delta.type == "text_delta":
                print(ts(), f"    text_delta       {event.delta.text!r}")
            elif event.delta.type == "input_json_delta":
                print(ts(), f"    input_json_delta {event.delta.partial_json!r}")
        elif event.type == "content_block_stop":
            print(ts(), f"■ content_block_stop   index={event.index}")
        elif event.type == "message_delta":
            print(ts(), f"■ message_delta        stop_reason={event.delta.stop_reason}")

    first_response = stream.get_final_message()

print("\n블록 구성:", [b.type for b in first_response.content])

   1238ms ▶ message_start
   1246ms ▶ content_block_start  index=0 type=text
   1246ms     text_delta       '서울의'
   1752ms     text_delta       ' 날씨와 미세먼지 상태를 함께 확인해보겠습니다.'
   1762ms ■ content_block_stop   index=0
   1763ms ▶ content_block_start  index=1 type=tool_use name=get_weather
   1763ms     input_json_delta ''
   2294ms     input_json_delta '{"cit'
   2298ms     input_json_delta 'y": "서울"}'
   2301ms ■ content_block_stop   index=1
   2304ms ▶ content_block_start  index=2 type=tool_use name=get_air_quality
   2305ms     input_json_delta ''
   2429ms     input_json_delta '{"city": "서울'
   2429ms     input_json_delta '"}'
   2429ms ■ content_block_stop   index=2
   2440ms ■ message_delta        stop_reason=tool_use

블록 구성: ['text', 'tool_use', 'tool_use']


위 출력에서 확인할 것 세 가지:

1. `index=0 type=text`의 `text_delta`들이 **먼저** 흘러나옵니다 → UI는 이 시점에 이미 텍스트를 렌더링 중
2. 텍스트 블록이 닫힌 뒤, `index=1 type=tool_use`가 **같은 스트림에서** 시작됩니다
3. 툴 입력 JSON도 통짜가 아니라 `input_json_delta` 조각으로 스트리밍됩니다

서울 날씨 + 미세먼지처럼 독립적인 요청이면 `tool_use` 블록이 **2개**(index=1, 2) 연달아 오는
병렬 호출도 볼 수 있습니다. 즉 "응답이 다 끝나고 나서야 툴 요청을 안다"가 아니라,
**스트림을 읽는 도중에 툴 요청의 시작과 내용을 실시간으로 알 수 있습니다.**

## 3. 실전 — 스트리밍 + 툴 실행 에이전틱 루프 (Claude Code 방식)

이제 실제 UX로 조립합니다. 각 라운드를 스트리밍으로 받아서:
- 텍스트는 나오는 즉시 출력 (사용자가 기다리지 않음)
- `tool_use` 블록이 시작되면 `⚙` 표시
- 스트림 종료 후 `get_final_message()`로 완성된 응답을 얻어 툴 실행 → 결과 반환 → 다음 라운드

스트리밍이어도 루프 구조는 논스트리밍과 동일합니다. `stop_reason`으로 종료를 판단합니다.

In [5]:
messages = [{"role": "user", "content": "서울이랑 부산 중에 오늘 야외 나들이하기 좋은 도시는 어디야? 날씨랑 공기질 다 보고 판단해줘."}]
MAX_ROUNDS = 10

for round_num in range(1, MAX_ROUNDS + 1):
    print(f"\n══════ 라운드 {round_num} (API 스트리밍 호출) ══════")

    with client.messages.stream(
        model=MODEL,
        max_tokens=8192,
        system=SYSTEM,
        tools=tools,
        messages=messages,
    ) as stream:
        for event in stream:
            if event.type == "content_block_delta" and event.delta.type == "text_delta":
                print(event.delta.text, end="", flush=True)  # 텍스트 실시간 출력
            elif event.type == "content_block_start" and event.content_block.type == "tool_use":
                print(f"\n  ⚙ 툴 요청 수신 중: {event.content_block.name}", flush=True)

        response = stream.get_final_message()

    if response.stop_reason != "tool_use":
        break  # 최종 답변 완료

    # 툴 실행 후 결과를 하나의 user 메시지로 반환
    messages.append({"role": "assistant", "content": response.content})
    tool_results = []
    for block in response.content:
        if block.type == "tool_use":
            result = TOOL_FUNCTIONS[block.name](**block.input)
            print(f"  ✔ {block.name}({json.dumps(block.input, ensure_ascii=False)}) → {result}")
            tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
    messages.append({"role": "user", "content": tool_results})

print(f"\n\n총 {round_num}번의 스트리밍 호출로 완료")


══════ 라운드 1 (API 스트리밍 호출) ══════
서울과 부산의 날씨와 미세먼지 수치를 모두 확인해서 비교해드리겠습니다.
  ⚙ 툴 요청 수신 중: get_weather

  ⚙ 툴 요청 수신 중: get_weather

  ⚙ 툴 요청 수신 중: get_air_quality

  ⚙ 툴 요청 수신 중: get_air_quality
  ✔ get_weather({"city": "서울"}) → {"city": "서울", "temp": 31, "sky": "맑음"}
  ✔ get_weather({"city": "부산"}) → {"city": "부산", "temp": 28, "sky": "흐림"}
  ✔ get_air_quality({"city": "서울"}) → {"city": "서울", "pm10": 45, "grade": "보통"}
  ✔ get_air_quality({"city": "부산"}) → {"city": "부산", "pm10": 22, "grade": "좋음"}

══════ 라운드 2 (API 스트리밍 호출) ══════
두 도시의 정보를 비교해보겠습니다.

## 오늘의 서울 vs 부산

| 항목 | 서울 | 부산 |
|------|------|------|
| 기온 | 31℃ | 28℃ |
| 하늘 | 맑음 ☀️ | 흐림 ☁️ |
| 미세먼지(PM10) | 45 (보통) | 22 (좋음) |

## 결론: **부산**을 추천합니다! 🌊

**이유는 이렇습니다.**

- **공기질**: 부산이 PM10 22(좋음)로, 서울(45, 보통)보다 훨씬 깨끗합니다. 야외에서 오래 활동하기엔 공기가 좋은 부산이 유리합니다.
- **기온**: 부산이 28℃로 서울(31℃)보다 3도 낮아 더위 부담이 덜합니다.
- **하늘**: 서울은 맑아서 화창하지만 그만큼 햇볕이 강하고 더울 수 있고, 부산은 흐려서 뙤약볕은 덜하지만 소나기 가능성은 살짝 염두에 두면 좋습니다.

**정리하면**, 화창한 풍경을 원하면 서울도 나쁘지 않지만, 야외 나들이의 쾌적

## 4. 정리 + 한 단계 더: eager input streaming

**정리하면** — "중간 스트리밍"의 정체는:
- 하나의 스트리밍 응답 안에서 텍스트 블록과 tool_use 블록이 **생성 순서대로** 내려온다
- 그래서 UI는 텍스트를 실시간 렌더링하다가, 스트림이 끝나기도 전에 "어떤 툴을 부르려는지" 알 수 있다
- 단, **툴을 실제 실행하고 결과를 돌려주는 건** 스트림이 끝난 뒤 → 다음 API 호출에서 (루프 구조는 동일)

**추가 옵션**: 기본적으로 툴 입력 JSON의 `input_json_delta`는 유효한 JSON 조각 단위로 버퍼링되어 내려옵니다.
툴 정의에 `"eager_input_streaming": true`를 주면 버퍼링 없이 생성되는 대로 더 잘게 스트리밍됩니다.
긴 인자(예: 파일 내용, 문서 본문)를 받는 툴에서 진행 상황을 실시간 표시하고 싶을 때 사용합니다.
아래 셀에서 메모 작성 툴로 확인해보세요 — Claude Code가 파일을 쓸 때 내용이 실시간으로 보이는 것도 이 방식입니다.

In [4]:
memo_tool = {
    "name": "save_memo",
    "description": "작성한 메모를 저장합니다.",
    "eager_input_streaming": True,  # 입력 JSON을 버퍼링 없이 즉시 스트리밍
    "input_schema": {
        "type": "object",
        "properties": {"text": {"type": "string", "description": "저장할 메모 본문"}},
        "required": ["text"],
    },
}

print("툴 입력이 실시간으로 흘러나오는 모습:\n")
with client.messages.stream(
    model=MODEL,
    max_tokens=8192,
    tools=[memo_tool],
    tool_choice={"type": "tool", "name": "save_memo"},
    messages=[{"role": "user", "content": "프롬프트 캐싱의 장점을 3문장으로 정리해서 메모로 저장해줘."}],
) as stream:
    for event in stream:
        if event.type == "content_block_delta" and event.delta.type == "input_json_delta":
            print(event.delta.partial_json, end="", flush=True)

    final = stream.get_final_message()

memo_input = next(b.input for b in final.content if b.type == "tool_use")
print("\n\n[파싱된 최종 입력]")
print(memo_input["text"])

툴 입력이 실시간으로 흘러나오는 모습:

{"text": "프롬프트 캐싱의 장점\n\n1. 반복되는 프롬프트(시스템 지침, 문서, 예시 등)를 캐싱해 재사용함으로써 API 호출 비용을 크게 절감할 수 있다.\n2. 캐시된 부분을 다시 처리하지 않아도 되므로 응답 지연 시간(레이턴시)이 줄어들어 사용자 경험이 향상된다.\n3. 긴 컨텍스트를 효율적으로 활용할 수 있어, 대용량 문서 기반 질의응답이나 반복 작업이 많은 애플리케이션에서 특히 유리하다."}

[파싱된 최종 입력]
프롬프트 캐싱의 장점

1. 반복되는 프롬프트(시스템 지침, 문서, 예시 등)를 캐싱해 재사용함으로써 API 호출 비용을 크게 절감할 수 있다.
2. 캐시된 부분을 다시 처리하지 않아도 되므로 응답 지연 시간(레이턴시)이 줄어들어 사용자 경험이 향상된다.
3. 긴 컨텍스트를 효율적으로 활용할 수 있어, 대용량 문서 기반 질의응답이나 반복 작업이 많은 애플리케이션에서 특히 유리하다.
